# Phase 4: Dashboard Data Preparation

## Objective

Prepare aggregated datasets and business metrics required for the Power BI dashboard — **CineMetrix**.

The outputs from this notebook will be exported to `../data/dashboard/` and imported directly into Power BI.

### Dashboard Structure (3 Pages)

| Page | Name | Purpose |
|------|------|---------|
| 1 | **Executive Overview** | High-level KPIs, content growth trend, rating distribution, Movies vs TV Shows, top genres, top countries, interactive slicers |
| 2 | **Global Footprint** | Geographic map, content leadership by country, drill-down analysis, country content composition by type |
| 3 | **Content Deep Dive** | Content evolution over time (Movie vs TV Show), top 7 genres donut chart, most frequent actors |

In [ ]:
import pandas as pd
import os

# Load cleaned datasets from Phase 2 & 3
df = pd.read_csv("../data/processed/netflix_clean.csv")
country_df = pd.read_csv("../data/processed/netflix_country_exploded.csv")
genre_df = pd.read_csv("../data/processed/netflix_genre_exploded.csv")

# Create output directory for dashboard-ready datasets
os.makedirs("../data/dashboard", exist_ok=True)

print(f"Main dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Country exploded: {country_df.shape[0]:,} rows")
print(f"Genre exploded: {genre_df.shape[0]:,} rows")
df.head()

---

## Dataset 1: KPI Summary

**Used on:** Page 1 — Executive Overview (KPI cards)

The top-level KPI cards display:
- **Countries** → 121
- **Genres** → 42
- **Total Movies** → 5K (5,377)
- **Total Titles** → 8K (7,787)
- **Total TV Shows** → 2K (2,410)

In [ ]:
kpi_summary = pd.DataFrame({
    "Metric": [
        "Total Titles",
        "Total Movies",
        "Total TV Shows",
        "Countries",
        "Genres"
    ],
    "Value": [
        len(df),
        (df["type"]=="Movie").sum(),
        (df["type"]=="TV Show").sum(),
        country_df["country"].nunique(),
        df["genres"].str.split(", ").explode().nunique()
    ]
})

kpi_summary

In [ ]:
kpi_summary.to_csv(
    "../data/dashboard/kpi_summary.csv",
    index=False
)

---

## Dataset 2: Yearly Titles

**Used on:** Page 1 — Executive Overview → "Content Growth Over Time" (area chart)

In [ ]:
yearly_titles = (
    df.groupby("release_year")
      .size()
      .reset_index(name="Titles")
)

yearly_titles.head()

In [ ]:
yearly_titles.to_csv(
    "../data/dashboard/yearly_titles.csv",
    index=False
)

---

## Dataset 3: Movies vs TV Shows Over Time

**Used on:**
- Page 1 — Executive Overview → "Movies vs TV Shows" (bar chart)
- Page 3 — Content Deep Dive → "Content Evolution Over Time" (line chart with Movie vs TV Show split)

In [ ]:
movies_tv = (
    df.groupby(["release_year","type"])
      .size()
      .reset_index(name="Titles")
)

movies_tv.head()

In [ ]:
movies_tv.to_csv(
    "../data/dashboard/movies_tv.csv",
    index=False
)

---

## Dataset 4: Country Summary

**Used on:**
- Page 1 — Executive Overview → "Top Countries" (horizontal bar chart)
- Page 2 — Global Footprint → "Netflix Content Across the World" (filled map)
- Page 2 — Global Footprint → "Content Leadership Across Countries" (bar chart)

In [ ]:
country_summary = (
    country_df["country"]
      .value_counts()
      .reset_index()
)

country_summary.columns = [
    "Country",
    "Titles"
]

country_summary.head(10)

In [ ]:
country_summary.to_csv(
    "../data/dashboard/country_summary.csv",
    index=False
)

---

## Dataset 5: Rating Summary

**Used on:** Page 1 — Executive Overview → "Content Rating Distribution" (treemap)

Also available as a slicer filter on the Executive Overview page.

In [ ]:
rating_summary = (
    df["rating"]
      .value_counts()
      .reset_index()
)

rating_summary.columns = [
    "Rating",
    "Titles"
]

rating_summary

In [ ]:
rating_summary.to_csv(
    "../data/dashboard/rating_summary.csv",
    index=False
)

---

## Dataset 6: Genre Summary

**Used on:**
- Page 1 — Executive Overview → "Top Genres" (horizontal bar chart)
- Page 3 — Content Deep Dive → "Top 7 Genres by Title Count" (donut chart)

Also available as a slicer filter on the Executive Overview page.

In [ ]:
genre_summary = (
    genre_df["genres"]
    .value_counts()
    .reset_index()
)

genre_summary.columns = [
    "Genre",
    "Titles"
]

genre_summary.head(10)

In [ ]:
genre_summary.to_csv(
    "../data/dashboard/genre_summary.csv",
    index=False
)

---

## Dataset 7: Country × Type Summary

**Used on:**
- Page 2 — Global Footprint → "Content Leadership Across Countries" (stacked/grouped bar by type)
- Page 2 — Global Footprint → "Country Content Composition" (stacked horizontal bar: Movie vs TV Show)
- Page 2 — Global Footprint → "Drill Down Analysis" (decomposition tree: type → country)

In [ ]:
country_type_summary = (
    country_df.groupby(["country", "type"])
    .size()
    .reset_index(name="Titles")
)

country_type_summary.head(10)

In [ ]:
country_type_summary.to_csv(
    "../data/dashboard/country_type_summary.csv",
    index=False
)

---

## Dataset 8: Top Cast Summary

**Used on:** Page 3 — Content Deep Dive → "Most Frequent Actors" (bar chart)

The dashboard displays the top 10 most frequently appearing cast members.

In [ ]:
top_cast_summary = (
    df["cast"]
    .str.split(", ")
    .explode()
    .value_counts()
    .head(15)
    .reset_index()
)
top_cast_summary.columns = ["Cast", "Appearances"]

top_cast_summary

In [ ]:
top_cast_summary.to_csv(
    "../data/dashboard/top_cast_summary.csv",
    index=False
)

---

## Data Validation

Cross-check exported values against the dashboard KPI cards to ensure consistency.

In [ ]:
# Expected values from the Power BI dashboard KPI cards
expected = {
    "Total Titles": 7787,
    "Total Movies": 5377,
    "Total TV Shows": 2410,
    "Countries": 121,
    "Genres": 42
}

print("Dashboard KPI Validation")
print("=" * 45)
all_pass = True
for _, row in kpi_summary.iterrows():
    metric = row["Metric"]
    actual = row["Value"]
    exp = expected.get(metric, "?")
    status = "✓" if actual == exp else "✗"
    if actual != exp:
        all_pass = False
    print(f"  {status} {metric}: {actual:,} (expected {exp:,})")

print("=" * 45)
print("All KPIs match!" if all_pass else "WARNING: Some KPIs differ!")

In [ ]:
# Validate dataset file counts
dashboard_files = os.listdir("../data/dashboard")
print(f"\nDashboard datasets exported: {len(dashboard_files)}")
for f in sorted(dashboard_files):
    size = os.path.getsize(f"../data/dashboard/{f}")
    print(f"  • {f} ({size:,} bytes)")

---

## Summary: Exported Datasets → Dashboard Page Mapping

| # | Dataset File | Dashboard Page | Visual |
|---|-------------|---------------|--------|
| 1 | `kpi_summary.csv` | Executive Overview | KPI Cards (Countries, Genres, Total Movies, Total Titles, Total TV Shows) |
| 2 | `yearly_titles.csv` | Executive Overview | Content Growth Over Time (area chart) |
| 3 | `movies_tv.csv` | Executive Overview + Content Deep Dive | Movies vs TV Shows (bar) / Content Evolution Over Time (line) |
| 4 | `country_summary.csv` | Executive Overview + Global Footprint | Top Countries (bar) / World Map / Content Leadership |
| 5 | `rating_summary.csv` | Executive Overview | Content Rating Distribution (treemap) |
| 6 | `genre_summary.csv` | Executive Overview + Content Deep Dive | Top Genres (bar) / Top 7 Genres (donut) |
| 7 | `country_type_summary.csv` | Global Footprint | Content Leadership / Country Content Composition / Drill Down Analysis |
| 8 | `top_cast_summary.csv` | Content Deep Dive | Most Frequent Actors (bar chart) |

### Interactive Filters (Slicers) on Executive Overview

| Slicer | Source |
|--------|--------|
| Type | `movies_tv.csv` → type column |
| Rating | `rating_summary.csv` → Rating column |
| Country | `country_summary.csv` → Country column |
| Genre | `genre_summary.csv` → Genre column |

These datasets will be imported into Power BI and used in the dashboard development phase (Phase 5).